In [1]:
import pandas as pd, numpy as np
import xgboost as xgb
import mlflow
from sklearn.metrics import average_precision_score, roc_auc_score

f = pd.read_parquet("data/features.parquet")

extra = (["TransactionID", "TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4",
          "card5", "card6", "addr1", "addr2", "dist1", "P_emaildomain", "R_emaildomain"]
         + [f"C{i}" for i in range(1, 15)]
         + [f"D{i}" for i in range(1, 16)]
         + [f"M{i}" for i in range(1, 10)])
raw = pd.read_csv("data/train_transaction.csv", usecols=extra)

df = f.merge(raw, on="TransactionID", how="left")
df = df.sort_values("event_timestamp").reset_index(drop=True)

cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"] + [f"M{i}" for i in range(1, 10)]
for c in cat_cols:
    df[c] = df[c].astype("category")

my_features = (["hour", "day_of_week", "tx_count_before", "secs_since_last", "amt_vs_avg",
                "device_changed", "addr_changed", "email_changed", "TransactionAmt",
                "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]
               + cat_cols
               + [f"C{i}" for i in range(1, 15)]
               + [f"D{i}" for i in range(1, 16)])

n = len(df)
a, b = int(n * 0.72), int(n * 0.80)
train, val, test = df.iloc[:a], df.iloc[a:b], df.iloc[b:]

X_train, y_train = train[my_features], train["isFraud"]
X_val, y_val = val[my_features], val["isFraud"]
X_test, y_test = test[my_features], test["isFraud"]

ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("features:", len(my_features), "| train:", len(train), "val:", len(val), "test:", len(test))

features: 59 | train: 425188 val: 47244 test: 118108


In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")

params = dict(n_estimators=1000, max_depth=6, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8,
              scale_pos_weight=ratio, tree_method="hist",
              enable_categorical=True, eval_metric="aucpr",
              early_stopping_rounds=50, random_state=42)

with mlflow.start_run(run_name="xgb_more_features"):
    model2 = xgb.XGBClassifier(**params)
    model2.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

    proba = model2.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)
    roc = roc_auc_score(y_test, proba)

    mlflow.log_params({k: v for k, v in params.items() if k != "enable_categorical"})
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("best_round", model2.best_iteration)

print("PR-AUC:", round(pr_auc, 4), " ROC-AUC:", round(roc, 4))
print("Stopped at round:", model2.best_iteration)

model2.save_model("models/xgb_v2.json")

[0]	validation_0-aucpr:0.33552
[50]	validation_0-aucpr:0.49007
[100]	validation_0-aucpr:0.52048
[150]	validation_0-aucpr:0.52777
[200]	validation_0-aucpr:0.53636
[250]	validation_0-aucpr:0.54314
[300]	validation_0-aucpr:0.54723
[350]	validation_0-aucpr:0.55329
[400]	validation_0-aucpr:0.55672
[450]	validation_0-aucpr:0.56200
[500]	validation_0-aucpr:0.56572
[550]	validation_0-aucpr:0.56740
[600]	validation_0-aucpr:0.57182
[650]	validation_0-aucpr:0.57315
[700]	validation_0-aucpr:0.57313
[750]	validation_0-aucpr:0.57529
[800]	validation_0-aucpr:0.57572
[850]	validation_0-aucpr:0.57722
[900]	validation_0-aucpr:0.57772
[950]	validation_0-aucpr:0.57999
[999]	validation_0-aucpr:0.58052
PR-AUC: 0.4727  ROC-AUC: 0.895
Stopped at round: 963
